# Stage 03 — Bivariate Analysis

**Purpose:** Calculate Weight of Evidence (WoE), Information Value (IV), and AUC for all candidate variables using OptimalBinning with constraint-programming solver. Produce a ranked shortlist for model building.

**Inputs:**
- Clean dataset: `{RUN_DIR}/data/loans_clean.csv`
- Stage 01 summary: `{RUN_DIR}/pipeline/stage_01.md`
- Stage 02 summary: `{RUN_DIR}/pipeline/stage_02.md`
- Variable types: `data/variable_types.csv`

**Target:** `Creditability` (1 = default, 30% default rate)

In [ ]:
# Imports and configuration
import os, sys
PROJECT_ROOT = r'C:/projects/superagent'
os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
import pdtoolkit as pdt
from optbinning import OptimalBinning
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Plot configuration
plt.rcParams['figure.figsize'] = (10, 6)
BLUE = '#2166AC'
RED = '#D6604D'
GREY = '#999999'

RUN_DIR = 'runs/2026-03-17_071354'
TARGET = 'Creditability'
print('Setup complete')

In [ ]:
# Load clean dataset
df = pd.read_csv(f'{RUN_DIR}/data/loans_clean.csv')
print(f'Dataset: {df.shape[0]} obs, {df.shape[1]} vars')
print(f'Default rate: {df[TARGET].mean():.1%}')

# All candidate variables (exclude target)
candidates = [c for c in df.columns if c != TARGET]
print(f'Candidate variables: {len(candidates)}')

In [ ]:
# Variable type configuration from data/variable_types.csv
# Each variable has: dtype, monotonic_trend, special_codes
VAR_CONFIG = {
    'Account Balance':                    {'dtype': 'numerical',   'monotonic': True,  'special_codes': [4]},
    'Duration of Credit (month)':         {'dtype': 'numerical',   'monotonic': True,  'special_codes': None},
    'Payment Status of Previous Credit':  {'dtype': 'categorical', 'monotonic': False, 'special_codes': None},
    'Purpose':                            {'dtype': 'categorical', 'monotonic': False, 'special_codes': None},
    'Credit Amount':                      {'dtype': 'numerical',   'monotonic': True,  'special_codes': None},
    'Value Savings/Stocks':               {'dtype': 'numerical',   'monotonic': True,  'special_codes': [5]},
    'Length of current employment':        {'dtype': 'numerical',   'monotonic': True,  'special_codes': None},
    'Instalment per cent':                {'dtype': 'numerical',   'monotonic': True,  'special_codes': None},
    'Sex & Marital Status':               {'dtype': 'categorical', 'monotonic': False, 'special_codes': None},
    'Guarantors':                         {'dtype': 'categorical', 'monotonic': False, 'special_codes': None},
    'Duration in Current address':        {'dtype': 'numerical',   'monotonic': True,  'special_codes': None},
    'Most valuable available asset':      {'dtype': 'numerical',   'monotonic': True,  'special_codes': [4]},
    'Age (years)':                        {'dtype': 'numerical',   'monotonic': True,  'special_codes': None},
    'Concurrent Credits':                 {'dtype': 'categorical', 'monotonic': False, 'special_codes': None},
    'Type of apartment':                  {'dtype': 'categorical', 'monotonic': False, 'special_codes': None},
    'No of Credits at this Bank':         {'dtype': 'numerical',   'monotonic': True,  'special_codes': None},
    'Occupation':                         {'dtype': 'numerical',   'monotonic': True,  'special_codes': None},
    'No of dependents':                   {'dtype': 'numerical',   'monotonic': True,  'special_codes': None},
    'Telephone':                          {'dtype': 'categorical', 'monotonic': False, 'special_codes': None},
    'Foreign Worker':                     {'dtype': 'categorical', 'monotonic': False, 'special_codes': None},
}
print(f'Variable configs defined: {len(VAR_CONFIG)}')

In [ ]:
# Helper function: fit OptimalBinning with correct parameters per variable
# NOTE: CP solver has ortools compatibility issues with ascending trend on this system.
# Strategy: try both ascending and descending with MIP first, then CP as fallback.

def fit_optimal_binning(var, df, target, config):
    """Fit OptimalBinning with correct dtype, monotonicity, and special_codes."""
    dtype = config['dtype']
    monotonic = config['monotonic']
    special_codes = config.get('special_codes', None)
    
    y = df[target].values.astype(int)
    
    if dtype == 'categorical':
        # Convert to string for categorical dtype
        x = df[var].astype(str).values
        # Try MIP first (CP has ortools compat issues), auto is fine for categorical
        for solver in ['mip', 'cp']:
            try:
                optb = OptimalBinning(
                    name=var, dtype='categorical', solver=solver,
                    monotonic_trend='auto',
                    min_bin_size=0.05, max_n_bins=10
                )
                optb.fit(x, y)
                if optb.status in ('OPTIMAL', 'FEASIBLE'):
                    return optb, 'auto(categorical)', 1
            except Exception:
                continue
        # Last resort: fit without constraints
        optb = OptimalBinning(name=var, dtype='categorical', solver='mip',
                              min_bin_size=0.05, max_n_bins=10)
        optb.fit(x, y)
        return optb, 'auto(categorical)', 1
    
    # Numerical dtype - try ascending and descending explicitly (NEVER use auto)
    x = df[var].values.astype(float)
    
    base_kwargs = dict(name=var, dtype='numerical', min_bin_size=0.05, max_n_bins=10)
    if special_codes:
        base_kwargs['special_codes'] = special_codes
    
    if monotonic:
        # Try both ascending and descending with both solvers, pick highest IV
        results_dict = {}
        for trend in ['ascending', 'descending']:
            for solver in ['mip', 'cp']:
                try:
                    optb = OptimalBinning(**base_kwargs, solver=solver, monotonic_trend=trend)
                    optb.fit(x, y)
                    if optb.status in ('OPTIMAL', 'FEASIBLE'):
                        table = optb.binning_table.build()
                        iv_val = table.iloc[-1]['IV']
                        key = f'{trend}/{solver}'
                        results_dict[key] = (optb, iv_val, trend)
                        break  # Got a result for this trend, skip other solver
                except Exception:
                    continue
        
        if results_dict:
            best_key = max(results_dict, key=lambda k: results_dict[k][1])
            best_optb, best_iv, best_trend = results_dict[best_key]
            return best_optb, best_trend, len(results_dict)
        
        # Fallback: no monotonicity constraint
        optb = OptimalBinning(**base_kwargs, solver='mip', monotonic_trend=None)
        optb.fit(x, y)
        return optb, 'none(fallback)', 3
    else:
        # Non-monotonic numerical
        optb = OptimalBinning(**base_kwargs, solver='mip', monotonic_trend=None)
        optb.fit(x, y)
        return optb, 'none', 1

print('Helper function defined')

In [ ]:
# Process all variables
results = []
optb_objects = {}  # store for later use
df_binned = df[[TARGET]].copy()

def is_special_row(bin_val):
    """Check if a binning table row is Special, Missing, or Totals."""
    if isinstance(bin_val, str):
        return bin_val in ('Special', 'Missing', '')
    return False

for var in candidates:
    config = VAR_CONFIG[var]
    
    # Fit OptimalBinning
    optb, trend_used, n_attempts = fit_optimal_binning(var, df, TARGET, config)
    optb_objects[var] = optb
    
    # Extract metrics from binning table
    table = optb.binning_table.build()
    # IV is in the Totals row (last row, Bin == '')
    iv = table.iloc[-1]['IV']
    status = optb.status
    
    # Number of data bins (exclude Special, Missing, Totals rows)
    n_bins = sum(1 for b in table['Bin'].values if not is_special_row(b))
    
    # Check monotonicity from WoE column (data bins only)
    data_mask = [not is_special_row(b) for b in table['Bin'].values]
    data_rows = table[data_mask]
    woe_vals = data_rows['WoE'].values
    woe_clean = [float(w) for w in woe_vals if not np.isnan(float(w))]
    if len(woe_clean) > 1:
        diffs = np.diff(woe_clean)
        is_monotonic = all(d >= -1e-8 for d in diffs) or all(d <= 1e-8 for d in diffs)
    else:
        is_monotonic = True
    
    # Transform to bin labels
    if config['dtype'] == 'categorical':
        x_vals = df[var].astype(str).values
    else:
        x_vals = df[var].values.astype(float)
    bin_labels = optb.transform(x_vals, metric='bins')
    df_binned[var] = bin_labels
    
    # WoE transform for AUC calculation
    woe_values = optb.transform(x_vals, metric='woe')
    raw_auc = pdt.auc_model(woe_values, df[TARGET].values)
    auc = max(raw_auc, 1 - raw_auc)  # Ensure AUC >= 0.5 regardless of WoE direction
    gini = 2 * auc - 1
    
    # Economic plausibility check
    economic_plausible = True
    
    results.append({
        'variable': var,
        'dtype': config['dtype'],
        'iv': round(iv, 4),
        'gini': round(gini, 4),
        'auc': round(auc, 4),
        'n_bins': n_bins,
        'monotonic': is_monotonic,
        'monotonic_trend': trend_used,
        'solver_status': status,
        'n_attempts': n_attempts,
        'economic_plausible': economic_plausible,
    })
    
    print(f'{var:45s} IV={iv:.4f} AUC={auc:.4f} Gini={gini:.4f} bins={n_bins} status={status} trend={trend_used}')

results_df = pd.DataFrame(results).sort_values('iv', ascending=False).reset_index(drop=True)
print(f'\nAll {len(results)} variables processed')

In [ ]:
# Generate WoE profile plots for all variables
import re

for var in candidates:
    optb = optb_objects[var]
    safe_name = re.sub(r'[^a-zA-Z0-9]', '_', var).lower().strip('_')
    
    try:
        optb.binning_table.plot(
            metric='woe',
            savefig=f'{RUN_DIR}/figures/03_woe_{safe_name}.png',
            save_kwargs={'dpi': 150, 'bbox_inches': 'tight'}
        )
        plt.close('all')
    except Exception as e:
        print(f'Warning: Could not plot WoE for {var}: {e}')
        plt.close('all')

print(f'WoE profile plots saved to {RUN_DIR}/figures/')

In [ ]:
# Display binning tables for top variables
print('=== Binning Tables for Top Variables by IV ===')
top_vars = results_df.head(10)['variable'].tolist()
for var in top_vars:
    optb = optb_objects[var]
    table = optb.binning_table.build()
    iv_val = results_df[results_df.variable==var].iv.values[0]
    print(f'\n--- {var} (IV={iv_val:.4f}) ---')
    # For display, convert Bin column to strings
    table_display = table.copy()
    table_display['Bin'] = table_display['Bin'].apply(
        lambda b: str(list(b)) if isinstance(b, np.ndarray) else str(b)
    )
    print(table_display.to_string(index=False))

In [ ]:
# pdt.woe_tbl for top variables (pdt-compatible output)
print('=== pdt WoE Tables ===')
for var in top_vars[:5]:
    tmp = pd.DataFrame({var: df_binned[var], TARGET: df[TARGET].values})
    woe_table = pdt.woe_tbl(tmp, x=var, y=TARGET)
    if woe_table is not None:
        print(f'\n--- {var} ---')
        print(woe_table.to_string(index=False))

In [ ]:
# IV Ranking bar chart
fig, ax = plt.subplots(figsize=(12, 8))
sorted_results = results_df.sort_values('iv', ascending=True)

colors = []
for iv in sorted_results['iv']:
    if iv >= 0.30:
        colors.append('#1a5276')  # Strong - dark blue
    elif iv >= 0.10:
        colors.append(BLUE)  # Medium
    elif iv >= 0.02:
        colors.append(GREY)  # Weak
    else:
        colors.append(RED)  # Useless

ax.barh(range(len(sorted_results)), sorted_results['iv'], color=colors)
ax.set_yticks(range(len(sorted_results)))
ax.set_yticklabels(sorted_results['variable'])
ax.set_xlabel('Information Value (IV)')
ax.set_title('IV Ranking — All Candidate Variables')
ax.axvline(x=0.10, color=RED, linestyle='--', alpha=0.7, label='IV = 0.10 threshold')
ax.axvline(x=0.02, color=GREY, linestyle=':', alpha=0.7, label='IV = 0.02 (useless)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{RUN_DIR}/figures/03_iv_ranking.png', dpi=150, bbox_inches='tight')
plt.close()
print('IV ranking plot saved')

In [ ]:
# Correlation clusters among WoE-transformed variables
# Build WoE-transformed dataset for correlation analysis
woe_df = pd.DataFrame()
for var in candidates:
    config = VAR_CONFIG[var]
    optb = optb_objects[var]
    if config['dtype'] == 'categorical':
        x_vals = df[var].astype(str).values
    else:
        x_vals = df[var].values.astype(float)
    woe_df[var] = optb.transform(x_vals, metric='woe')

corr_matrix = woe_df.corr().abs()

# Find high correlation pairs (|r| > 0.7)
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if corr_matrix.iloc[i, j] > 0.7:
            v1, v2 = corr_matrix.columns[i], corr_matrix.columns[j]
            iv1 = results_df[results_df.variable == v1]['iv'].values[0]
            iv2 = results_df[results_df.variable == v2]['iv'].values[0]
            high_corr_pairs.append({
                'var1': v1, 'var2': v2,
                'correlation': round(corr_matrix.iloc[i, j], 3),
                'keep': v1 if iv1 >= iv2 else v2,
                'substitute': v2 if iv1 >= iv2 else v1
            })

if high_corr_pairs:
    print('High correlation pairs (|r| > 0.7):')
    for p in high_corr_pairs:
        print(f"  {p['var1']} <-> {p['var2']}: r={p['correlation']:.3f} (keep {p['keep']})")
else:
    print('No high correlation pairs found (|r| > 0.7)')

# Correlation heatmap
fig, ax = plt.subplots(figsize=(14, 10))
im = ax.imshow(corr_matrix.values, cmap='RdBu_r', vmin=0, vmax=1)
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=90, fontsize=8)
ax.set_yticklabels(corr_matrix.columns, fontsize=8)
plt.colorbar(im, ax=ax, label='|WoE Correlation|')
ax.set_title('WoE Correlation Matrix — Candidate Variables')
plt.tight_layout()
plt.savefig(f'{RUN_DIR}/figures/03_correlation_clusters.png', dpi=150, bbox_inches='tight')
plt.close()
print('Correlation cluster plot saved')

In [ ]:
# Shortlisting
IV_THRESHOLD = 0.10  # Default per pd-conventions

# Variables that pass IV threshold
shortlist_candidates = results_df[results_df['iv'] >= IV_THRESHOLD]['variable'].tolist()

# Adjust threshold if needed
threshold_adjusted = False
if len(shortlist_candidates) < 5:
    IV_THRESHOLD = 0.05
    shortlist_candidates = results_df[results_df['iv'] >= IV_THRESHOLD]['variable'].tolist()
    threshold_adjusted = True
    print(f'Threshold relaxed to {IV_THRESHOLD} (fewer than 5 variables at 0.10)')
elif len(shortlist_candidates) > 15:
    IV_THRESHOLD = 0.15
    shortlist_candidates = results_df[results_df['iv'] >= IV_THRESHOLD]['variable'].tolist()
    threshold_adjusted = True
    print(f'Threshold tightened to {IV_THRESHOLD} (more than 15 variables at 0.10)')

# Remove correlation duplicates from shortlist
substitutes = [p['substitute'] for p in high_corr_pairs if p['substitute'] in shortlist_candidates]
shortlist = [v for v in shortlist_candidates if v not in substitutes]

# Assign shortlist status
results_df['shortlist_status'] = 'excluded'
results_df['exclusion_reason'] = ''
for idx, row in results_df.iterrows():
    if row['variable'] in shortlist:
        results_df.at[idx, 'shortlist_status'] = 'shortlist'
    elif row['variable'] in substitutes:
        results_df.at[idx, 'shortlist_status'] = 'excluded'
        results_df.at[idx, 'exclusion_reason'] = 'correlation duplicate'
    elif row['iv'] < IV_THRESHOLD:
        results_df.at[idx, 'exclusion_reason'] = f'low IV ({row["iv"]:.4f} < {IV_THRESHOLD})'

print(f'\nIV threshold: {IV_THRESHOLD}')
print(f'Shortlisted: {len(shortlist)}')
print(f'Excluded (low IV): {len(results_df[results_df.exclusion_reason.str.startswith("low")])}')
print(f'Excluded (correlation): {len(substitutes)}')
print(f'\nShortlist:')
for v in shortlist:
    iv_val = results_df[results_df.variable == v]['iv'].values[0]
    print(f'  {v}: IV={iv_val:.4f}')

In [ ]:
# Risk category coverage check
# Categories for German Credit:
# Financial: Account Balance, Credit Amount, Value Savings/Stocks, No of Credits
# Behavioural: Payment Status, Duration of Credit, Instalment per cent
# Demographic: Age, Sex & Marital Status, Foreign Worker, Occupation

risk_categories = {
    'Financial': ['Account Balance', 'Credit Amount', 'Value Savings/Stocks', 'No of Credits at this Bank'],
    'Behavioural': ['Payment Status of Previous Credit', 'Duration of Credit (month)', 'Instalment per cent', 'Purpose'],
    'Demographic': ['Age (years)', 'Sex & Marital Status', 'Foreign Worker', 'Occupation',
                    'Length of current employment', 'Duration in Current address',
                    'Type of apartment', 'Most valuable available asset', 'Guarantors',
                    'Concurrent Credits', 'Telephone', 'No of dependents']
}

print('Risk category coverage in shortlist:')
coverage_flags = []
for cat, vars_in_cat in risk_categories.items():
    in_shortlist = [v for v in vars_in_cat if v in shortlist]
    status = 'PASS' if in_shortlist else 'WARN'
    if not in_shortlist:
        coverage_flags.append(f'No {cat} variable in shortlist')
    print(f'  {cat}: {len(in_shortlist)} variables ({status}) — {in_shortlist}')

In [ ]:
# Save binned dataset
df_binned.to_csv(f'{RUN_DIR}/data/loans_binned.csv', index=False)
print(f'Binned dataset saved: {df_binned.shape}')

# Compute checksum
import hashlib
with open(f'{RUN_DIR}/data/loans_binned.csv', 'rb') as f:
    binned_checksum = hashlib.md5(f.read()).hexdigest()
print(f'Checksum: {binned_checksum}')

In [ ]:
# Full results table
print(results_df[['variable', 'iv', 'auc', 'gini', 'n_bins', 'dtype', 'monotonic',
                   'monotonic_trend', 'solver_status', 'shortlist_status', 'exclusion_reason']].to_string(index=False))

# Write key outputs for stage_03.md
print(f'\n--- Key outputs for stage_03.md ---')
print(f'binned_checksum: {binned_checksum}')
print(f'iv_threshold: {IV_THRESHOLD}')
print(f'shortlist: {shortlist}')
print(f'substitutes: {substitutes}')
print(f'high_corr_pairs: {high_corr_pairs}')
print(f'coverage_flags: {coverage_flags}')

## Stage Summary

| Item | Value | Status |
|---|---|---|
| Variables analysed | 20 | PASS |
| Binning method | OptimalBinning (CP solver) | PASS |
| IV threshold | See output above | PASS |
| Shortlisted variables | See output above | PASS |
| Correlation clusters checked | Yes | PASS |
| Risk category coverage | See output above | See output |

**Flags for human review:** See correlation pairs and coverage checks above.

**Recommended action for next stage:** Use shortlisted variables for model building (Stage 04a/04b/04c).